# Heaps — The Array-Backed Priority Structure

A **binary heap** is a complete binary tree stored compactly in an array, satisfying the **heap property**: in a *max-heap* every parent is $\ge$ its children (a min-heap flips the inequality). Completeness lets the tree live in an array with no pointers, where a node at index $i$ finds its relatives by arithmetic. This gives $O(1)$ access to the extreme element and $O(\log n)$ insert and delete — the basis of priority queues and heapsort. Every operation records snapshots so the sift up/down can be watched in both the array and the tree.

$$ \text{parent}(i) = \left\lfloor \tfrac{i-1}{2} \right\rfloor, \qquad \text{left}(i) = 2i+1, \qquad \text{right}(i) = 2i+2. $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=650)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

def parent(i): return (i-1)//2
def left(i):   return 2*i+1
def right(i):  return 2*i+2

# Position nodes of a complete tree of size n: level by level, centered.
def heap_layout(n):
    pos = {}
    if n == 0: return pos
    levels = int(np.floor(np.log2(n))) + 1
    for i in range(n):
        depth = int(np.floor(np.log2(i+1)))
        idx_in_level = (i+1) - 2**depth
        count_in_level = 2**depth
        x = (idx_in_level + 0.5) / count_in_level
        pos[i] = (x * (2**(levels-1)) * 2, -depth)
    return pos

def draw_heap(arr, hi=(), swap=(), title='', note=''):
    n = len(arr)
    pos = heap_layout(n)
    fig, (axt, axa) = plt.subplots(2, 1, figsize=(9, 6), gridspec_kw={'height_ratios':[3,1]})
    # tree edges
    for i in range(n):
        if i in pos:
            for ch in (left(i), right(i)):
                if ch < n:
                    x0,y0 = pos[i]; x1,y1 = pos[ch]
                    axt.plot([x0,x1],[y0,y1], color='lightgray', lw=1.5, zorder=1)
    for i in range(n):
        x,y = pos[i]
        if i in swap:   c = 'tomato'
        elif i in hi:   c = 'gold'
        else:           c = 'lightsteelblue'
        axt.add_patch(plt.Circle((x,y), 0.34, facecolor=c, edgecolor='k', zorder=2))
        axt.text(x, y, str(arr[i]), ha='center', va='center', fontsize=11, zorder=3)
    if pos:
        xs=[p[0] for p in pos.values()]; ys=[p[1] for p in pos.values()]
        axt.set_xlim(min(xs)-0.8, max(xs)+0.8); axt.set_ylim(min(ys)-0.7, 0.7)
    axt.set_title(title); axt.axis('off')
    # array row
    for i in range(n):
        if i in swap:   c = 'tomato'
        elif i in hi:   c = 'gold'
        else:           c = 'lightsteelblue'
        axa.add_patch(plt.Rectangle((i,0),0.95,1, facecolor=c, edgecolor='k'))
        axa.text(i+0.475, 0.5, str(arr[i]), ha='center', va='center', fontsize=10)
        axa.text(i+0.475, -0.35, str(i), ha='center', va='center', fontsize=7, color='gray')
    axa.set_xlim(0, max(n,1)); axa.set_ylim(-0.6, 1.1)
    axa.set_title(note, fontsize=10); axa.axis('off')
    plt.tight_layout(); plt.show()

## A Tree Without Pointers

Because the heap is a *complete* tree, level-order indexing lets parents and children be reached by arithmetic alone — no stored links. Pick an index and the widget highlights its parent and children, the only navigation a heap ever needs.

$$ \text{children of } i = \{2i+1,\ 2i+2\}, \qquad \text{parent of } i = \left\lfloor \tfrac{i-1}{2}\right\rfloor. $$

In [ ]:
demo_arr = [42, 29, 18, 14, 7, 11, 9, 3, 2, 4]

def show_relatives(i):
    rel = set()
    p = parent(i) if i > 0 else None
    if p is not None: rel.add(p)
    kids = {c for c in (left(i), right(i)) if c < len(demo_arr)}
    draw_heap(demo_arr, hi=kids | ({p} if p is not None else set()), swap={i},
              title=f'index {i}: parent={p}, children={sorted(kids)}',
              note='red = selected, gold = parent & children')

i_s = widgets.IntSlider(value=3, min=0, max=len(demo_arr)-1, description='index i')
display(i_s, widgets.interactive_output(show_relatives, {'i': i_s}))

IntSlider(value=3, description='index i', max=9)

Output()

## Insertion — Append, Then Sift Up

A new key is appended at the end (preserving completeness) and then **sifted up**: while it exceeds its parent, the two swap, bubbling toward the root until the heap property is restored. The number of swaps is at most the height, $O(\log n)$.

$$ \text{while } a[i] > a[\text{parent}(i)]:\ \text{swap}; \ i \leftarrow \text{parent}(i). $$

In [3]:
def insert_frames(base, key):
    a = list(base); frames = []
    a.append(key); i = len(a)-1
    frames.append((list(a), {i}, set(), f'append {key} at index {i}'))
    while i > 0 and a[i] > a[parent(i)]:
        p = parent(i)
        frames.append((list(a), set(), {i, p}, f'{a[i]} > {a[p]}: swap up'))
        a[i], a[p] = a[p], a[i]; i = p
        frames.append((list(a), {i}, set(), f'moved to index {i}'))
    frames.append((list(a), set(), set(), 'heap property restored'))
    return frames

base_heap = [42, 29, 18, 14, 7, 11, 9]
ik_s = widgets.IntSlider(value=33, min=1, max=99, description='insert key')
ins_area = widgets.Output()

def relaunch_ins(*_):
    frames = insert_frames(base_heap, ik_s.value)
    def draw(k):
        arr, hi, sw, note = frames[k]
        draw_heap(arr, hi=hi, swap=sw, title=f'insert {ik_s.value} . step {k}/{len(frames)-1}',
                  note=note)
    ins_area.clear_output(wait=True)
    with ins_area: make_player(len(frames), draw)
ik_s.observe(relaunch_ins, 'value')
display(ik_s, ins_area)
relaunch_ins()

IntSlider(value=33, description='insert key', max=99, min=1)

Output()

## Extract-Max — Swap to End, Then Sift Down

The maximum sits at the root. To remove it, swap the root with the last element, drop the last (the old max), then **sift down**: repeatedly swap the new root with its larger child until neither child exceeds it. Again $O(\log n)$.

$$ \text{swap } a[0] \leftrightarrow a[n-1];\ \text{pop};\ \text{sift down from root.} $$

In [4]:
def sift_down_frames(a, frames, n_active):
    i = 0
    while True:
        l, r = left(i), right(i)
        largest = i
        if l < n_active and a[l] > a[largest]: largest = l
        if r < n_active and a[r] > a[largest]: largest = r
        cand = {c for c in (l, r) if c < n_active}
        frames.append((list(a[:n_active]), cand, set(), f'compare {a[i]} with children'))
        if largest == i:
            frames.append((list(a[:n_active]), set(), set(), 'no swap: heap restored'))
            break
        frames.append((list(a[:n_active]), set(), {i, largest},
                       f'swap with larger child {a[largest]}'))
        a[i], a[largest] = a[largest], a[i]; i = largest

def extract_frames(base):
    a = list(base); frames = []
    frames.append((list(a), {0}, set(), f'max = {a[0]} at root'))
    n = len(a)
    a[0], a[n-1] = a[n-1], a[0]
    frames.append((list(a), set(), {0, n-1}, f'swap root with last (index {n-1})'))
    removed = a.pop(); n -= 1
    frames.append((list(a), set(), set(), f'removed {removed}; sift down new root'))
    sift_down_frames(a, frames, n)
    return frames

ext_base = [42, 29, 18, 14, 7, 11, 9, 3, 2]
ext_area = widgets.Output()
def show_ext():
    frames = extract_frames(ext_base)
    def draw(k):
        arr, hi, sw, note = frames[k]
        draw_heap(arr, hi=hi, swap=sw, title=f'extract-max . step {k}/{len(frames)-1}', note=note)
    with ext_area: make_player(len(frames), draw)
display(ext_area)
show_ext()

Output()

## Heapify — Building a Heap in $O(n)$

A naive build inserts $n$ keys one by one for $O(n\log n)$. **Floyd's build-heap** instead sift-downs every internal node from the last parent up to the root, which is provably $O(n)$ because most nodes are shallow. Step through to watch sub-heaps form bottom-up and merge into one.

$$ \text{for } i = \left\lfloor \tfrac n2 \right\rfloor - 1 \text{ down to } 0:\ \text{sift-down}(i), \qquad T(n) = O(n). $$

In [5]:
def build_heap_frames(values):
    a = list(values); n = len(a); frames = []
    frames.append((list(a), set(), set(), 'start (arbitrary array)'))
    start = n//2 - 1
    for root in range(start, -1, -1):
        frames.append((list(a), {root}, set(), f'sift-down from index {root}'))
        i = root
        while True:
            l, r = left(i), right(i); largest = i
            if l < n and a[l] > a[largest]: largest = l
            if r < n and a[r] > a[largest]: largest = r
            if largest == i: break
            frames.append((list(a), set(), {i, largest}, f'swap {a[i]} <-> {a[largest]}'))
            a[i], a[largest] = a[largest], a[i]; i = largest
    frames.append((list(a), set(), set(), 'valid max-heap'))
    return frames

bh_seed = widgets.IntSlider(value=4, min=0, max=20, description='seed')
bh_n = widgets.IntSlider(value=10, min=4, max=15, description='n')
bh_area = widgets.Output()
def relaunch_bh(*_):
    rng = np.random.default_rng(bh_seed.value)
    vals = list(rng.integers(1, 99, bh_n.value))
    frames = build_heap_frames(vals)
    def draw(k):
        arr, hi, sw, note = frames[k]
        draw_heap(arr, hi=hi, swap=sw, title=f'build-heap . step {k}/{len(frames)-1}', note=note)
    bh_area.clear_output(wait=True)
    with bh_area: make_player(len(frames), draw)
bh_seed.observe(relaunch_bh, 'value'); bh_n.observe(relaunch_bh, 'value')
display(widgets.HBox([bh_n, bh_seed]), bh_area)
relaunch_bh()

Output()

## Heapsort — Repeated Extraction Sorts In Place

With a max-heap built, repeatedly swapping the root to the current end and shrinking the heap by one leaves the array sorted ascending — all in place, $O(n\log n)$ overall. The sorted tail (green) grows from the right while the heap region shrinks. Step through to watch the boundary move.

$$ T(n) = \underbrace{O(n)}_{\text{build}} + \underbrace{n \cdot O(\log n)}_{\text{extractions}} = O(n \log n). $$

In [6]:
def heapsort_frames(values):
    a = list(values); n = len(a); frames = []
    # build
    for root in range(n//2 - 1, -1, -1):
        i = root
        while True:
            l,r = left(i), right(i); largest = i
            if l < n and a[l] > a[largest]: largest = l
            if r < n and a[r] > a[largest]: largest = r
            if largest == i: break
            a[i],a[largest]=a[largest],a[i]; i=largest
    frames.append((list(a), n, set(), set(), 'heap built; now extract repeatedly'))
    end = n
    while end > 1:
        a[0], a[end-1] = a[end-1], a[0]
        end -= 1
        frames.append((list(a), end, set(), {0}, f'move max to index {end} (sorted tail grows)'))
        i = 0
        while True:
            l,r = left(i), right(i); largest = i
            if l < end and a[l] > a[largest]: largest = l
            if r < end and a[r] > a[largest]: largest = r
            if largest == i: break
            a[i],a[largest]=a[largest],a[i]
            frames.append((list(a), end, set(), {i, largest}, 'sift down within shrinking heap'))
            i = largest
    frames.append((list(a), 0, set(), set(), f'sorted: {a}'))
    return frames

hs_seed = widgets.IntSlider(value=2, min=0, max=20, description='seed')
hs_area = widgets.Output()
def relaunch_hs(*_):
    rng = np.random.default_rng(hs_seed.value)
    vals = list(rng.integers(1, 99, 9))
    frames = heapsort_frames(vals)
    def draw(k):
        arr, heap_end, hi, sw, note = frames[k]
        n = len(arr)
        # sorted tail (indices >= heap_end) shown green via custom draw
        pos = heap_layout(heap_end)
        fig, (axt, axa) = plt.subplots(2,1, figsize=(9,6), gridspec_kw={'height_ratios':[3,1]})
        for i in range(heap_end):
            for ch in (left(i), right(i)):
                if ch < heap_end:
                    x0,y0=pos[i]; x1,y1=pos[ch]; axt.plot([x0,x1],[y0,y1], color='lightgray', lw=1.5, zorder=1)
        for i in range(heap_end):
            x,y=pos[i]
            c = 'tomato' if i in sw else ('gold' if i in hi else 'lightsteelblue')
            axt.add_patch(plt.Circle((x,y),0.34, facecolor=c, edgecolor='k', zorder=2))
            axt.text(x,y,str(arr[i]), ha='center', va='center', fontsize=11, zorder=3)
        if pos:
            xs=[p[0] for p in pos.values()]; ys=[p[1] for p in pos.values()]
            axt.set_xlim(min(xs)-0.8,max(xs)+0.8); axt.set_ylim(min(ys)-0.7,0.7)
        axt.set_title(f'heapsort . step {k}/{len(frames)-1} . heap size {heap_end}'); axt.axis('off')
        for i in range(n):
            if i >= heap_end: c='seagreen'
            elif i in sw: c='tomato'
            elif i in hi: c='gold'
            else: c='lightsteelblue'
            axa.add_patch(plt.Rectangle((i,0),0.95,1, facecolor=c, edgecolor='k'))
            axa.text(i+0.475,0.5,str(arr[i]), ha='center', va='center', fontsize=10)
        axa.set_xlim(0,n); axa.set_ylim(-0.2,1.1); axa.set_title(note, fontsize=10); axa.axis('off')
        plt.tight_layout(); plt.show()
    hs_area.clear_output(wait=True)
    with hs_area: make_player(len(frames), draw)
hs_seed.observe(relaunch_hs, 'value')
display(hs_seed, hs_area)
relaunch_hs()

IntSlider(value=2, description='seed', max=20)

Output()